# CR Impact Analyzer
## Notebook 4 — LLM Evaluation & Model Comparison

### Install evaluation libraries

In [ ]:
# Run this cell once to install evaluation dependencies
import subprocess
subprocess.run(["pip", "install", "rouge-score", "bert-score", "pandas", "matplotlib", "seaborn"], 
               capture_output=True)
print(" Evaluation libraries ready")

### Configuration

> **Note:** `EMBEDDING_MODEL` below must be the same model used to build the vector store in Notebook 2, otherwise the query fails with an embedding-dimension mismatch. Notebook 2 currently uses `all-mpnet-base-v2` — either rebuild the vector store with `all-MiniLM-L6-v2` before running this notebook, or change the value below to match.

In [ ]:
import os

BASE_DIR         = os.getcwd()
VECTOR_STORE_DIR = os.path.join(BASE_DIR, "vector_store")
COLLECTION_NAME  = "system_docs"
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"   # must match the model the vector store was built with (Notebook 2)
OLLAMA_BASE_URL  = "http://localhost:11434"
TOP_K            = 6

MODELS_TO_EVALUATE = [
    "llama3",
    "mistral",
     "phi3",
     "gemma:7b",
    "qwen2.5:14b",
    "phi4",
    "mistral-nemo",
    "gemma2:9b"
]

print(" Config ready")
print(f"   Models to evaluate: {MODELS_TO_EVALUATE}")

###  Step 1 — Define your evaluation dataset


In [ ]:
EVAL_DATASET = [
    {
        "id": "Cr_ID",
        "cr": (
            " "
        ),
        "reference": (
            " "
        ),
    },
    {
        "id": "Cr_ID",
        "cr": (
            " "
        ),
        "reference": (
            " "
        ),
    },
    {
        "id": "Cr_ID",
        "cr": (
            " "
        ),
        "reference": (
            " "
        ),
    },
    {
        "id": "Cr_ID",
        "cr": (
            " "
        ),
        "reference": (
            " "
        ),
    },
]

print(f" Evaluation dataset ready — {len(EVAL_DATASET)} PDFs defined")
for item in EVAL_DATASET:
    print(f"    {item['id']}: {item['cr'][:60]}...")


###  Step 2 — Load vector store & helper functions

In [ ]:
import chromadb
import requests
import json
import time
from sentence_transformers import SentenceTransformer

# Load ChromaDB
print("Loading vector store...")
client      = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection  = client.get_collection(COLLECTION_NAME)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f" Vector store loaded — {collection.count()} chunks")


def retrieve(query, top_k=TOP_K):
    q_emb = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "text":      doc,
            "source":    meta["source"],
            "relevance": round(1 - dist, 4),
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


PROMPT_TEMPLATE = """You are a senior software architect specialising in impact assessment.

System documentation:

{context}


Change Request: {cr}

Provide a structured analysis covering:
1. IMPACT ANALYSIS - affected components, risk level, complexity
2. DEPENDENCIES - internal, external, data, API dependencies
3. REQUIRED ACTIONS - implementation steps, testing scope
4. RISKS & MITIGATION - top risks and mitigation strategies
"""

def call_llm(cr, context, model):
    prompt  = PROMPT_TEMPLATE.format(context=context, cr=cr)
    payload = {"model": model, "prompt": prompt, "stream": False}
    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload, timeout=300
    )
    resp.raise_for_status()
    answer   = resp.json().get("response", "")
    duration = round(time.time() - t0, 2)
    return answer, duration


print(" Helper functions ready")

###  Step 3 — Run all models on all CRs

In [ ]:
results_raw = []  # stores all (model, cr_id, answer, context, duration)

for model in MODELS_TO_EVALUATE:
    print(f"\n{'='*60}")
    print(f"  Model: {model}")
    print(f"{'='*60}")

    for item in EVAL_DATASET:
        print(f"  Running CR {item['id']}: {item['cr'][:50]}...")

        # Retrieve context
        hits    = retrieve(item["cr"])
        context = "\n\n".join([h["text"] for h in hits])
        avg_relevance = round(sum(h["relevance"] for h in hits) / len(hits), 4)

        # Call LLM
        try:
            answer, duration = call_llm(item["cr"], context, model)
            status = "✅"
        except Exception as e:
            answer, duration = f"ERROR: {e}", 0
            status = "❌"

        results_raw.append({
            "model":         model,
            "cr_id":         item["id"],
            "cr":            item["cr"],
            "reference":     item["reference"],
            "answer":        answer,
            "context":       context,
            "hits":          hits,
            "avg_relevance": avg_relevance,
            "duration_sec":  duration,
        })
        print(f"  {status}  Done in {duration}s")

print(f"\n Collected {len(results_raw)} responses total")

### Step 4 — Score each response

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import warnings
warnings.filterwarnings("ignore")

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


# Checks: how many sentences in the answer can be traced back to the context?
def score_faithfulness(answer, context):
    sentences = [s.strip() for s in answer.split(".") if len(s.strip()) > 20]
    if not sentences:
        return 0.0
    context_lower = context.lower()
    supported = 0
    for sent in sentences:
        # Check if key words from the sentence appear in context
        words = [w.lower() for w in sent.split() if len(w) > 4]
        if not words:
            continue
        matches = sum(1 for w in words if w in context_lower)
        if matches / len(words) >= 0.3:   # 30% of key words found in context
            supported += 1
    return round(supported / len(sentences), 4)


# Checks: does the answer cover the key topics of the CR?
def score_answer_relevancy(cr, answer):
    cr_words    = set(w.lower() for w in cr.split() if len(w) > 3)
    answer_lower = answer.lower()
    covered = sum(1 for w in cr_words if w in answer_lower)
    return round(covered / max(len(cr_words), 1), 4)


# Average relevance score of retrieved chunks
def score_context_precision(hits):
    if not hits:
        return 0.0
    return round(sum(h["relevance"] for h in hits) / len(hits), 4)


# Checks if the 4 required sections are present in the answer
def score_structure(answer):
    required_sections = [
        "impact",
        "dependenc",
        "action",
        "risk",
    ]
    answer_lower = answer.lower()
    found = sum(1 for s in required_sections if s in answer_lower)
    return round(found / len(required_sections), 4)


print("Scoring responses...\n")

# ROUGE-L (batch)
all_answers    = [r["answer"]    for r in results_raw]
all_references = [r["reference"] for r in results_raw]

# BERTScore (batch — runs once for efficiency)
print("  Computing BERTScore (this may take a minute)...")
P, R, F1 = bert_score(all_answers, all_references, lang="en", verbose=False)
bert_scores = F1.tolist()

# Add all scores to results
scored_results = []
for i, r in enumerate(results_raw):
    rouge_score_val = rouge.score(r["answer"], r["reference"])["rougeL"].fmeasure

    scored_results.append({
        **r,
        "faithfulness":      score_faithfulness(r["answer"], r["context"]),
        "answer_relevancy":  score_answer_relevancy(r["cr"], r["answer"]),
        "context_precision": score_context_precision(r["hits"]),
        "structure_score":   score_structure(r["answer"]),
        "rouge_l":           round(rouge_score_val, 4),
        "bert_score":        round(bert_scores[i], 4),
        "word_count":        len(r["answer"].split()),
    })

print(" All responses scored")

In [ ]:
clean_answers = []
clean_refs = []

for a, r in zip(all_answers, all_references):
    if not isinstance(a, str) or not isinstance(r, str):
        continue
        
    a = a.strip()
    r = r.strip()
    
    if len(a) < 2 or len(r) < 2:
        continue
        
    clean_answers.append(a)
    clean_refs.append(r)

print("After cleaning:", len(clean_answers))

### Step 5 — Results table

In [ ]:
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

df = pd.DataFrame(scored_results)

# Display columns
display_cols = [
    "model", "cr_id",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "structure_score",
    "rouge_l",
    "bert_score",
    "word_count",
    "duration_sec",
]

display(df[display_cols].sort_values(["model", "cr_id"]))

### Step 6 — Model comparison summary

In [ ]:
# Compute a weighted overall score for each model
# Weights reflect what matters most for CR analysis
WEIGHTS = {
    "faithfulness":      0.30,   # most important — no hallucination
    "answer_relevancy":  0.25,   # answers the actual CR
    "structure_score":   0.20,   # all 4 sections present
    "bert_score":        0.15,   # semantic quality
    "rouge_l":           0.10,   # text overlap
}

summary_rows = []
for model in MODELS_TO_EVALUATE:
    model_df = df[df["model"] == model]
    row = {"model": model}
    overall = 0
    for metric, weight in WEIGHTS.items():
        avg = model_df[metric].mean()
        row[metric] = round(avg, 3)
        overall += avg * weight
    row["avg_speed_sec"]  = round(model_df["duration_sec"].mean(), 1)
    row["avg_word_count"] = round(model_df["word_count"].mean())
    row["overall_score"]  = round(overall, 3)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("overall_score", ascending=False)

print("\n" + "="*60)
print("  MODEL COMPARISON SUMMARY")
print("="*60)
display(summary_df)

best_model = summary_df.iloc[0]["model"]
best_score = summary_df.iloc[0]["overall_score"]
print(f"\n  Best model for your CR analysis: '{best_model}'  (score: {best_score:.3f})")

###  Step 7 — Visualize results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np

matplotlib.rcParams["figure.dpi"] = 120
sns.set_theme(style="darkgrid")

metrics    = list(WEIGHTS.keys())
models     = summary_df["model"].tolist()
n_models   = len(models)
n_metrics  = len(metrics)
x          = np.arange(n_metrics)
bar_width  = 0.8 / n_models

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors = sns.color_palette("husl", n_models)
for i, model in enumerate(models):
    row    = summary_df[summary_df["model"] == model].iloc[0]
    values = [row[m] for m in metrics]
    offset = (i - n_models / 2) * bar_width + bar_width / 2
    bars   = ax.bar(x + offset, values, bar_width, label=model, color=colors[i])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in metrics], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score (0–1)")
ax.set_title("Metric Scores by Model", fontsize=13, fontweight="bold")
ax.legend(title="Model")

ax2 = axes[1]
overall_scores = summary_df["overall_score"].tolist()
speeds         = summary_df["avg_speed_sec"].tolist()
bar_colors     = [colors[i] for i in range(n_models)]

bars = ax2.barh(models, overall_scores, color=bar_colors, height=0.4)
ax2.set_xlim(0, 1.1)
ax2.set_xlabel("Overall Score (weighted)")
ax2.set_title("Overall Score & Speed per Model", fontsize=13, fontweight="bold")

for bar, speed in zip(bars, speeds):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f"{bar.get_width():.3f}  ({speed}s)",
             va="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "model_evaluation.png"), bbox_inches="tight")
plt.show()
print("\n Chart saved to model_evaluation.png")

###  Step 8 — Read full answers for any CR

In [ ]:
INSPECT_MODEL = "llama3"    # model name
INSPECT_CR_ID = "CR-001"    # cr id from your dataset

row = df[(df["model"] == INSPECT_MODEL) & (df["cr_id"] == INSPECT_CR_ID)]

if row.empty:
    print(f"No result found for model='{INSPECT_MODEL}' cr_id='{INSPECT_CR_ID}'")
else:
    r = row.iloc[0]
    print(f"Model  : {r['model']}")
    print(f"CR     : {r['cr']}")
    print(f"Scores : faithfulness={r['faithfulness']}  relevancy={r['answer_relevancy']}  "
          f"structure={r['structure_score']}  rouge={r['rouge_l']}  bert={r['bert_score']}")
    print(f"Speed  : {r['duration_sec']}s   |   Words: {r['word_count']}")
    print("\n" + "="*70)
    print("ANSWER:")
    print("="*70)
    print(r["answer"])

### Step 9 — Save full evaluation report

In [ ]:
import datetime

output_dir = os.path.join(BASE_DIR, "evaluations")
os.makedirs(output_dir, exist_ok=True)
timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Save CSV
csv_path = os.path.join(output_dir, f"eval_results_{timestamp}.csv")
df[display_cols].to_csv(csv_path, index=False)

# Save summary CSV
summary_path = os.path.join(output_dir, f"eval_summary_{timestamp}.csv")
summary_df.to_csv(summary_path, index=False)

# Save text report
report_path = os.path.join(output_dir, f"eval_report_{timestamp}.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"CR IMPACT ANALYZER — EVALUATION REPORT\n")
    f.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Models evaluated: {MODELS_TO_EVALUATE}\n")
    f.write(f"CRs evaluated: {len(EVAL_DATASET)}\n\n")
    f.write("=" * 60 + "\n")
    f.write("SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string())
    f.write(f"\n\n Best model: {best_model}  (overall score: {best_score:.3f})\n")
    f.write("\n" + "=" * 60 + "\n")
    f.write("DETAILED RESULTS\n")
    f.write("=" * 60 + "\n\n")
    for r in scored_results:
        f.write(f"Model: {r['model']}  |  {r['cr_id']}: {r['cr']}\n")
        f.write(f"Scores — faithfulness:{r['faithfulness']}  relevancy:{r['answer_relevancy']}  "
                f"structure:{r['structure_score']}  rouge:{r['rouge_l']}  bert:{r['bert_score']}\n")
        f.write(f"Speed: {r['duration_sec']}s  |  Words: {r['word_count']}\n")
        f.write("Answer:\n" + r['answer'] + "\n")
        f.write("-" * 60 + "\n\n")

print(f" Evaluation report saved:")
print(f"   CSV     : {csv_path}")
print(f"   Summary : {summary_path}")
print(f"   Report  : {report_path}")

---

| Metric | Good score | What low score means |
|---|---|---|
| **Faithfulness** | > 0.70 | Model is hallucinating — answer not grounded in docs |
| **Answer Relevancy** | > 0.70 | Answer doesn't address the CR question |
| **Context Precision** | > 0.75 | Retrieved chunks aren't relevant enough |
| **Structure Score** | = 1.00 | One or more of the 4 sections is missing |
| **ROUGE-L** | > 0.30 | Low text overlap with expected answer |
| **BERTScore** | > 0.85 | Semantically different from expected answer |
| **Speed** | < 30s | Model too slow for practical use |

### Weight explanation
The **overall score** weights metrics by importance for CR analysis:
- Faithfulness 30% — hallucination is the biggest risk
- Answer Relevancy 25% — must address the actual CR
- Structure 20% — all 4 sections must be present
- BERTScore 15% — semantic quality matters
- ROUGE-L 10% — least important for open-ended analysis